In [6]:
import requests
import pandas as pd
import time
import re
import os
from datetime import datetime

TRUDVSEM_URL = "https://opendata.trudvsem.ru/api/v1/vacancies"
QUERIES_TV = [
    {"text": "разработчик программист",   "label": "Разработчик"},
    {"text": "product manager",           "label": "Product Manager"},
    {"text": "data scientist",            "label": "Data Scientist"},
    {"text": "технический директор CTO",  "label": "CTO"},
    {"text": "аналитик данных",           "label": "Аналитик"},
    {"text": "менеджер по развитию",      "label": "BD Manager"},
]

def fetch_trudvsem(query, limit=50):
    try:
        r = requests.get(TRUDVSEM_URL,
                         params={"text": query, "limit": limit, "offset": 0},
                         timeout=15)
        r.raise_for_status()
        data = r.json()
        return data.get("results", {}).get("vacancies", [])
    except Exception as e:
        print(f" ошибка {e}")
        return []

def parse_trudvsem(v, label):
    vac = v.get("vacancy") or v
    sal_from = vac.get("salary_min")
    sal_to   = vac.get("salary_max")
    avg = round((sal_from + sal_to) / 2) if sal_from and sal_to else (sal_from or sal_to)
    return {
        "label":       label,
        "name":        vac.get("job-name", ""),
        "company":     (vac.get("company") or {}).get("name", ""),
        "salary_from": sal_from,
        "salary_to":   sal_to,
        "salary_avg":  avg,
        "region":      vac.get("region", {}).get("region_name", "") if isinstance(vac.get("region"), dict) else "",
        "collected_at": datetime.now().strftime("%Y-%m-%d"),
    }

tv_rows = []
for q in QUERIES_TV:
    print(f" Trudvsem: {q['text']}")
    vacancies = fetch_trudvsem(q["text"])
    for v in vacancies:
        tv_rows.append(parse_trudvsem(v, q["label"]))
    print(f" Получили: {len(vacancies)}")
    time.sleep(0.5)


df_salary = pd.DataFrame(tv_rows)
print(f"\nДанные по зарплатам: {len(df_salary)} строк")
display(df_salary.head())

 Trudvsem: разработчик программист
 Получили: 38
 Trudvsem: product manager
 Получили: 25
 Trudvsem: data scientist
 Получили: 5
 Trudvsem: технический директор CTO
 Получили: 1
 Trudvsem: аналитик данных
 Получили: 50
 Trudvsem: менеджер по развитию
 Получили: 50

Данные по зарплатам: 169 строк


,label,name,company,salary_from,salary_to,salary_avg,region,collected_at
0,Разработчик,Программист/разработчик,ИП Алиев Руслан Чингизович,200000,250000,225000,,2026-05-24
1,Разработчик,Программист-разработчик,"ООО ""УК ""РУСМОЛКО""",100000,100000,100000,,2026-05-24
2,Разработчик,Программист-разработчик,"АО ""САЯНСКХИМПЛАСТ""",128000,129000,128500,,2026-05-24
3,Разработчик,1C Программист / Разработчик,"ПАО ПЗ ""СИГНАЛ""",120000,120000,120000,,2026-05-24
4,Разработчик,программист разработчик РЭА,"ООО НПФ ""МЕТА""",50000,50000,50000,,2026-05-24
